# Exploratory Data Analysis (EDA) on Engineered Features
This notebook analyzes the `data/feature_engineered_dataset.csv` generated in Week 1.
The goal is to understand the correlation of our rolling temporal features with machine failure, visualize distributions, and dry-run SMOTE to inspect synthetic data points.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Data Loading & Basic Inspection

In [ ]:
df = pd.read_csv('../data/feature_engineered_dataset.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
failure_counts = df['Machine failure'].value_counts(normalize=True) * 100
print('Class Distribution (%):\n', failure_counts)

## 2. Feature Correlation Analysis
Let's see which features (raw vs 1H rolling vs 8H ema) have the strongest linear correlation with `Machine failure`.

In [ ]:
drop_cols = ['UDI', 'Product ID', 'Type', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df_numeric = df.drop(columns=drop_cols)
correlations = df_numeric.corr()['Machine failure'].sort_values(ascending=False)
print('Top 10 Positively Correlated Features:')
print(correlations.head(10))
print('\nTop 5 Negatively Correlated Features:')
print(correlations.tail(5))

In [ ]:
plt.figure(figsize=(10, 6))
# Exclude the target itself
top_corr = correlations.drop('Machine failure').head(10)
sns.barplot(x=top_corr.values, y=top_corr.index, palette='viridis')
plt.title('Top 10 Features Correlated with Machine Failure')
plt.xlabel('Pearson Correlation')
plt.show()

## 3. Distribution Comparison (Healthy vs. Failure)
Visualizing the separation between classes for the strongest features.

In [ ]:
features_to_plot = ['Torque [Nm]', 'Tool wear [min]_roll_mean_480', 'Air temperature [K]_ema_60']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(features_to_plot):
    if feat in df.columns:
        sns.kdeplot(data=df, x=feat, hue='Machine failure', fill=True, ax=axes[i], common_norm=False)
        axes[i].set_title(f'Distribution of {feat}')
plt.tight_layout()
plt.show()

## 4. SMOTE Dry-Run & Visualization
We will apply SMOTE to the dataset and use PCA (Principal Component Analysis) to project the multi-dimensional sensor space into 2D. This allows us to visually inspect where the synthetic failure points are being generated.

In [ ]:
X = df_numeric.drop(columns=['Machine failure'])
y = df_numeric['Machine failure']
smote = SMOTE(sampling_strategy=2/3, random_state=42)
X_res, y_res = smote.fit_resample(X, y)
print(f'Original target distribution: {y.value_counts().to_dict()}')
print(f'SMOTE target distribution: {y_res.value_counts().to_dict()}')

In [ ]:
pca = PCA(n_components=2)
# PCA on original data
X_pca_orig = pca.fit_transform(StandardScaler().fit_transform(X))
# PCA on SMOTE data
X_pca_res = pca.transform(StandardScaler().fit_transform(X_res))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=X_pca_orig[:,0], y=X_pca_orig[:,1], hue=y, alpha=0.5, ax=axes[0], palette=['blue', 'red'])
axes[0].set_title('PCA Layout - Original Data (Extreme Imbalance)')

sns.scatterplot(x=X_pca_res[:,0], y=X_pca_res[:,1], hue=y_res, alpha=0.2, ax=axes[1], palette=['blue', 'orange'])
axes[1].set_title('PCA Layout - After SMOTE (Synthetic points in Orange)')

plt.show()